# AIO-Mesh Rev1 — explicit input package

This notebook prepares a command plan by default. The user selects the archive, supplies its separately recorded SHA-256, and explicitly enables a phase. Training is new training; evaluation requires a compatible evaluation package and artifact preflight. The historical notebook is separate.


In [ ]:
from pathlib import Path

PACKAGE_PATH = ""  # exact attached ZIP path
EXPECTED_PACKAGE_SHA256 = ""  # SHA-256 printed by package_inputs.py
WORK_ROOT = Path("/kaggle/working/rev1-input")  # must not already exist
PHASE = "train"  # "train" or "evaluate"; must match bundle mode
TOPOLOGY = "vietnam"
RUN_ID = "user-run-001"
RUN_PHASE = False  # set True only when ready to execute

# Explicit new-training settings; fill in before planning a train phase.
TRAIN_SEED = None
MAIN_STEPS = None
ABLATION_STEPS = None
N_ENVS = None
ROLLOUT_STEPS = None
TRAIN_SCENARIO = None
EVAL_STEPS = 3000


In [ ]:
import hashlib
import os
import re
import subprocess
import sys
import tempfile
import zipfile
from pathlib import PurePosixPath

if not PACKAGE_PATH or not re.fullmatch(r"[0-9a-f]{64}", EXPECTED_PACKAGE_SHA256):
    raise ValueError("Specify an exact package path and lowercase SHA-256")
package = Path(PACKAGE_PATH)
if not package.is_file() or WORK_ROOT.exists():
    raise ValueError("Package must exist and WORK_ROOT must be new")
digest = hashlib.sha256()
with package.open("rb") as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b""):
        digest.update(chunk)
if digest.hexdigest() != EXPECTED_PACKAGE_SHA256:
    raise ValueError("Input archive SHA-256 mismatch")
WORK_ROOT.parent.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryDirectory(dir=WORK_ROOT.parent) as temporary:
    with zipfile.ZipFile(package) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)):
            raise ValueError("Duplicate archive entries")
        for name in names:
            parts = PurePosixPath(name).parts
            if not parts or parts[0] != "CoreRouter" or ".." in parts or "\\" in name:
                raise ValueError("Unsafe archive path")
        archive.extractall(temporary)
    extracted = Path(temporary) / "CoreRouter"
    environment = dict(os.environ)
    environment["PYTHONPATH"] = str(extracted)
    checker = ("import json,sys; from scripts.rev1.package_inputs import verify_package; "
               "m=verify_package(sys.argv[1],sys.argv[2]); print(json.dumps(m))")
    verified = subprocess.run([sys.executable, "-c", checker, str(package.resolve()),
                               EXPECTED_PACKAGE_SHA256], cwd=extracted, env=environment,
                              check=True, capture_output=True, text=True)
    import json
    manifest = json.loads(verified.stdout)
    expected_mode = {"train": "training", "evaluate": "evaluation"}.get(PHASE)
    if manifest["mode"] != expected_mode:
        raise ValueError("Package mode does not match requested phase")
    extracted.rename(WORK_ROOT)
print("Verified package:", manifest["protocol_id"], manifest["mode"])


In [ ]:
import re
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]*", RUN_ID):
    raise ValueError("RUN_ID must contain only letters, digits, underscore or hyphen")
output = "results/rev1/" + RUN_ID
command = [sys.executable, "scripts/rev1/phase_driver.py", "--stage", PHASE,
           "--topology", TOPOLOGY, "--output-dir", output]
if PHASE == "evaluate":
    command.extend(["--input-package", str(package.resolve()),
                    "--input-sha256", EXPECTED_PACKAGE_SHA256])
if PHASE == "train":
    settings = [TRAIN_SEED, MAIN_STEPS, ABLATION_STEPS, N_ENVS, ROLLOUT_STEPS, TRAIN_SCENARIO]
    if any(value is None for value in settings):
        raise ValueError("Set all explicit training settings")
    for option, value in zip(["--train-seed", "--main-steps", "--ablation-steps",
                              "--n-envs", "--rollout-steps", "--train-scenario"], settings):
        command.extend([option, str(value)])
else:
    command.extend(["--steps", str(EVAL_STEPS)])
if RUN_PHASE:
    command.append("--execute")
subprocess.run(command, cwd=WORK_ROOT, check=True)


The driver covers training, per-scenario ablation and main benchmark, load sweep, and a separate cross-topology coverage audit. Evaluation verifies the same archive hash and resolved protocol before artifact preflight. Save outputs and model/normalizer files; do not interpret a plan or smoke as a research result.
